<a href="https://colab.research.google.com/github/yichungcheng/SSL_MSM_Endoscope_Spine_surgery/blob/main/00_SSLMSM_baseline_msn_closer_tf_step_debug_smoketest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSN-style TensorFlow baseline (closer to paper math)

This notebook rewrites the original baseline to better match the paper's Section 3.1:

- uses **CLS token** as image representation
- adds **learnable prototype matrix Q**
- computes `softmax(Q z / tau)`
- uses **different student/teacher temperatures**
- adds **me-max regularizer**
- adds **patch masking on anchor views**
- supports **multiple anchor views M**
- keeps **EMA teacher**

Still simplified relative to the full paper/release:

- no Sinkhorn-Knopp target sharpening
- masking is random masking only by default
- training loop is kept compact for readability


In [ ]:
# 1. 設置與安裝函式庫
!pip install -q torch torchvision
!pip install -q segmentation_models_pytorch
!pip install -q albumentations # 數據增強庫
!pip install -q pycocotools # 處理 COCO 格式的官方工具，用於解析多邊形
!pip install tensorflow

# 2. 匯入必要的函式庫
import os
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pycocotools.coco import COCO
from google.colab import drive
import shutil

# 3. 連接 Google Drive
drive.mount('/content/gdrive')

# 4. 定義路徑
# 您的資料夾名稱
DRIVE_DIR = '/content/gdrive/MyDrive/PMBM/論文程式/MSM'
# DATASET_NAME = 'Endoscope Spine Surgery.v12i.coco'
# UNZIP_PATH = '/content/dataset'


# DATA_ROOT = DRIVE_DIR # 假設數據已經在 Drive 資料夾內

# DRIVE_DIR = '/content/gdrive/MyDrive/PMBM/論文程式/MSM'
# MODULE_DIR = os.path.join(DRIVE_DIR, 'ssl_msn')
# os.makedirs(MODULE_DIR, exist_ok=True)
# MODEL_SAVE_DIR = os.path.join(MODULE_DIR, 'models')
# os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


# 檢查路徑結構
print(f"數據根目錄: {DRIVE_DIR}")

import os
import sys
import gc
import time
import importlib
import numpy as np
import tensorflow as tf
os.environ['TZ'] = 'Asia/Taipei'
time.tzset()
# 測試印出現在時間
print("現在的台灣時間是:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time())))
# ============================================================
# 0. Paths
# ============================================================
# DRIVE_DIR = '/content/gdrive/MyDrive/PMBM/論文程式/MSM'
MODULE_DIR = os.path.join(DRIVE_DIR, 'ssl_msn_paper_closer_tf')
print(f"MODULE_DIR根目錄: {MODULE_DIR}")
os.makedirs(MODULE_DIR, exist_ok=True)

MODEL_SAVE_DIR = os.path.join(MODULE_DIR, 'models')
print(f"MODEL_SAVE_DIR根目錄: {MODEL_SAVE_DIR}")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

DATA_ROOT = DRIVE_DIR
IMAGE_DIR = f"{DATA_ROOT}/cholec80/cholec80_extracted/frames"
print(f"Image根目錄: {IMAGE_DIR}")

# ============================================================
# 1. Device
# ============================================================
gpus = tf.config.list_physical_devices('GPU')
DEVICE = 'GPU' if gpus else 'CPU'
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print(f"Could not enable memory growth: {e}")
print(f"TensorFlow device mode: {DEVICE}")

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

# # ============================================================
# # 2. dataset.py original
# # ============================================================
# with open(os.path.join(MODULE_DIR, 'dataset.py'), 'w') as f:
#     f.write('''import tensorflow as tf
# import os
# import numpy as np

# IMG_SIZE = 224

# # Global list to store problematic image paths
# problematic_image_paths = []

# # 2026/04/13 當讀取異常image，則skipped
# def _parse_image_py(path_tensor):
#     """Read and decode one image path.

#     Returns:
#         image: float32 array of shape (IMG_SIZE, IMG_SIZE, 3)
#         valid: bool, False when file is unreadable/corrupted and should be skipped.
#     """
#     path = path_tensor.numpy().decode("utf-8")
#     try:
#         img_bytes = tf.io.read_file(path).numpy()
#         if not img_bytes:
#             print(f"Warning: {path} is empty or could not be read. Skipping.")
#             problematic_image_paths.append(path)
#             return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32), np.array(False, dtype=np.bool_)

#         img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False).numpy()
#         img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE)).numpy()
#         img = img.astype(np.float32) / 255.0
#         return img, np.array(True, dtype=np.bool_)
#     except Exception as e:
#         print(f"Error processing {path}: {e}. Skipping.")
#         problematic_image_paths.append(path)
#         return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32), np.array(False, dtype=np.bool_)


# def parse_image(path):
#     img, valid = tf.py_function(
#         _parse_image_py,
#         inp=[path],
#         Tout=[tf.float32, tf.bool],
#     )
#     img.set_shape((IMG_SIZE, IMG_SIZE, 3))
#     valid.set_shape(())
#     return img, valid


# def build_dataset(frames_root, batch_size, split_type=None):
#     """
#     frames_root/
#       ├── Video01/*.jpg
#       ├── Video02/*.jpg
#       └── ...
#     """
#     global problematic_image_paths
#     problematic_image_paths = []

#     split_ranges = {
#         "train": range(1, 49),
#         "validation": range(49, 60),
#         "test": range(60, 81),
#     }

#     if split_type and split_type in split_ranges:
#         video_indices = split_ranges[split_type]
#     else:
#         video_indices = range(1, 81)

#     all_image_paths = []
#     for i in video_indices:
#         video_dir = os.path.join(frames_root, f"video{i:02d}")
#         if not os.path.isdir(video_dir):
#             print(f"Warning: video directory not found: {video_dir}")
#             continue

#         for filename in os.listdir(video_dir):
#             if filename.lower().endswith((".jpg", ".jpeg", ".png")):
#                 all_image_paths.append(os.path.join(video_dir, filename))

#     print(f"[INFO] Total frames loaded for {split_type or 'all'}: {len(all_image_paths)}")

#     if not all_image_paths:
#         print("No image paths found. Returning an empty dataset.")
#         empty = tf.zeros((0, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32)
#         return tf.data.Dataset.from_tensor_slices(empty).batch(batch_size).prefetch(tf.data.AUTOTUNE)

#     ds = tf.data.Dataset.from_tensor_slices(all_image_paths)
#     ds = ds.shuffle(buffer_size=2048)
#     ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)

#     # Skip unreadable/corrupted images.
#     ds = ds.filter(lambda img, valid: valid)
#     ds = ds.map(lambda img, valid: img, num_parallel_calls=tf.data.AUTOTUNE)

#     ds = ds.batch(batch_size)
#     ds = ds.prefetch(tf.data.AUTOTUNE)
#     return ds

# ''')

#============================================================
# 2. dataset.py new one
# ============================================================
with open(os.path.join(MODULE_DIR, 'dataset.py'), 'w') as f:
    f.write('''import tensorflow as tf
import os
import numpy as np

IMG_SIZE = 224

# Global list to store problematic image paths
problematic_image_paths = []

def _parse_image_py(path_tensor):
    """Read and decode one image path.

    Returns:
        path: original image path (bytes/string)
        image: float32 array of shape (IMG_SIZE, IMG_SIZE, 3)
        valid: bool, False when file is unreadable/corrupted and should be skipped.
        orig_h: original image height before resize
        orig_w: original image width before resize
        orig_c: original image channels before resize
    """
    path = path_tensor.numpy().decode("utf-8")
    try:
        img_bytes = tf.io.read_file(path).numpy()
        if not img_bytes:
            print(f"Warning: {path} is empty or could not be read. Skipping.")
            problematic_image_paths.append(path)
            return (
                path.encode("utf-8"),
                np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32),
                np.array(False, dtype=np.bool_),
                np.int32(-1),
                np.int32(-1),
                np.int32(-1),
            )

        img = tf.image.decode_image(
            img_bytes,
            channels=3,
            expand_animations=False
        ).numpy()

        if img.ndim != 3:
            print(f"Warning: decoded image rank != 3 for {path}, shape={img.shape}. Skipping.")
            problematic_image_paths.append(path)
            return (
                path.encode("utf-8"),
                np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32),
                np.array(False, dtype=np.bool_),
                np.int32(-1),
                np.int32(-1),
                np.int32(-1),
            )

        orig_h, orig_w, orig_c = img.shape

        # 額外保險：如果 decode 後 channel 不是 3，也跳過
        if orig_c != 3:
            print(f"Warning: decoded image channel != 3 for {path}, shape={img.shape}. Skipping.")
            problematic_image_paths.append(path)
            return (
                path.encode("utf-8"),
                np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32),
                np.array(False, dtype=np.bool_),
                np.int32(orig_h),
                np.int32(orig_w),
                np.int32(orig_c),
            )

        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE)).numpy()
        img = img.astype(np.float32) / 255.0

        return (
            path.encode("utf-8"),
            img,
            np.array(True, dtype=np.bool_),
            np.int32(orig_h),
            np.int32(orig_w),
            np.int32(orig_c),
        )

    except Exception as e:
        print(f"Error processing {path}: {e}. Skipping.")
        problematic_image_paths.append(path)
        return (
            path.encode("utf-8"),
            np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32),
            np.array(False, dtype=np.bool_),
            np.int32(-1),
            np.int32(-1),
            np.int32(-1),
        )


def parse_image(path):
    path_out, img, valid, orig_h, orig_w, orig_c = tf.py_function(
        _parse_image_py,
        inp=[path],
        Tout=[tf.string, tf.float32, tf.bool, tf.int32, tf.int32, tf.int32],
    )

    path_out.set_shape(())
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    valid.set_shape(())
    orig_h.set_shape(())
    orig_w.set_shape(())
    orig_c.set_shape(())

    return path_out, img, valid, orig_h, orig_w, orig_c


def _debug_shape(path, img, orig_h, orig_w, orig_c):
    """在進 augment 前先印出 shape 與 path。"""
    tf.print(
        "[DEBUG before augment]",
        "path =", path,
        "| resized_shape =", tf.shape(img),
        "| orig_shape =", [orig_h, orig_w, orig_c],
        summarize=-1
    )

    tf.debugging.assert_rank(img, 3, message="Image rank is not 3 before augment")
    tf.debugging.assert_equal(
        tf.shape(img)[-1], 3,
        message="Image channel is not 3 before augment"
    )

    return path, img


def build_dataset(frames_root, batch_size, split_type=None, debug_before_augment=False):
    """
    frames_root/
      ├── video01/*.jpg
      ├── video02/*.jpg
      └── ...
    Returns:
        ds yielding (path_batch, image_batch)
    """
    global problematic_image_paths
    problematic_image_paths = []

    split_ranges = {
        "train": range(1, 49),
        "validation": range(49, 60),
        "test": range(60, 81),
    }

    if split_type and split_type in split_ranges:
        video_indices = split_ranges[split_type]
    else:
        video_indices = range(1, 81)

    all_image_paths = []
    for i in video_indices:
        video_dir = os.path.join(frames_root, f"video{i:02d}")
        if not os.path.isdir(video_dir):
            print(f"Warning: video directory not found: {video_dir}")
            continue

        for filename in sorted(os.listdir(video_dir)):
            if filename.lower().endswith((".jpg", ".jpeg", ".png")):
                all_image_paths.append(os.path.join(video_dir, filename))

    print(f"[INFO] Total frames loaded for {split_type or 'all'}: {len(all_image_paths)}")

    if not all_image_paths:
        print("No image paths found. Returning an empty dataset.")
        empty_paths = tf.constant([], dtype=tf.string)
        empty_imgs = tf.zeros((0, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32)
        return tf.data.Dataset.from_tensor_slices((empty_paths, empty_imgs)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    ds = tf.data.Dataset.from_tensor_slices(all_image_paths)
    ds = ds.shuffle(buffer_size=min(2048, len(all_image_paths)))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)

    # 先跳過壞圖
    ds = ds.filter(lambda path, img, valid, orig_h, orig_w, orig_c: valid)

    # 如果要 debug，保留 path 並在 augment 前印資訊
    if debug_before_augment:
        ds = ds.map(
            lambda path, img, valid, orig_h, orig_w, orig_c: _debug_shape(path, img, orig_h, orig_w, orig_c),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    else:
        ds = ds.map(
            lambda path, img, valid, orig_h, orig_w, orig_c: (path, img),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds
''')


# # ============================================================
# # 3. augment.py original
# # ============================================================
# with open(os.path.join(MODULE_DIR, 'augment.py'), 'w', encoding='utf-8') as f:
#     f.write(r"""import tensorflow as tf

# IMG_SIZE = 224

# def strong_augment(x):
#     tf.print("DEBUG augment input shape =", tf.shape(x))
#     tf.debugging.assert_rank(x, 3)
#     tf.debugging.assert_equal(tf.shape(x)[-1], 3)
#     x = tf.convert_to_tensor(x)
#     x = tf.cast(x, tf.float32)

#     # 若資料是 0~255，先縮放到 0~1
#     x = tf.cond(
#         tf.reduce_max(x) > 1.0,
#         lambda: x / 255.0,
#         lambda: x
#     )

#     # 確保至少是 rank-3
#     tf.debugging.assert_greater_equal(tf.rank(x), 2, message="Input image rank must be >= 2")

#     # [H, W] -> [H, W, 1]
#     if x.shape.rank == 2:
#         x = tf.expand_dims(x, axis=-1)

#     # 動態處理 channel
#     if x.shape.rank == 3:
#         if x.shape[-1] == 1:
#             x = tf.image.grayscale_to_rgb(x)
#         elif x.shape[-1] != 3:
#             raise ValueError(f"Unsupported channel size: {x.shape[-1]}")
#     else:
#         # graph mode 下保底檢查
#         tf.debugging.assert_equal(tf.rank(x), 3, message="Input image must be rank 3 after preprocessing")

#     x = tf.ensure_shape(x, [None, None, 3])

#     # augmentation
#     x = tf.image.random_flip_left_right(x)
#     x = tf.image.random_brightness(x, 0.2)
#     x = tf.image.random_contrast(x, 0.6, 1.4)
#     x = tf.image.random_saturation(x, 0.6, 1.4)

#     crop_size = tf.random.uniform([], int(IMG_SIZE * 0.8), IMG_SIZE + 1, dtype=tf.int32)
#     x = tf.image.random_crop(x, size=[crop_size, crop_size, 3])
#     x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
#     x = tf.clip_by_value(x, 0.0, 1.0)
#     return x

# def target_view(batch):
#     batch = tf.cast(batch, tf.float32)
#     return tf.map_fn(strong_augment, batch, fn_output_signature=tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32))
#     # return tf.map_fn(strong_augment, batch, fn_output_signature=tf.float32)
# def anchor_views(batch, num_views=2):
#     batch = tf.cast(batch, tf.float32)
#     views = []
#     for _ in range(num_views):
#         views.append(
#             tf.map_fn(
#                 strong_augment,
#                 batch,
#                 fn_output_signature=tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32)
#             )
#         )
#     return views
# """)

# ============================================================
# 3. augment.py new
# ============================================================
with open(os.path.join(MODULE_DIR, 'augment.py'), 'w', encoding='utf-8') as f:
    f.write(r"""import tensorflow as tf

IMG_SIZE = 224

def strong_augment(path, x):
    path = tf.convert_to_tensor(path)
    x = tf.convert_to_tensor(x)
    x = tf.cast(x, tf.float32)

    tf.print("[AUGMENT INPUT] path =", path, "| shape =", tf.shape(x), summarize=-1)

    # 若資料是 0~255，先縮放到 0~1
    x = tf.cond(
        tf.reduce_max(x) > 1.0,
        lambda: x / 255.0,
        lambda: x
    )

    # 強制檢查 rank
    tf.debugging.assert_rank(x, 3, message="Input image rank must be 3 before augment")

    # 若 channel 不為 3，直接報錯並附上 path
    tf.debugging.assert_equal(
        tf.shape(x)[-1],
        3,
        message="Input image channel must be 3 before augment"
    )

    x = tf.ensure_shape(x, [None, None, 3])

    # augmentation
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.2)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = tf.image.random_saturation(x, 0.6, 1.4)

    crop_size = tf.random.uniform(
        [],
        minval=int(IMG_SIZE * 0.8),
        maxval=IMG_SIZE + 1,
        dtype=tf.int32
    )

    tf.print("[AUGMENT CROP] path =", path, "| crop_size =", crop_size, "| pre_crop_shape =", tf.shape(x), summarize=-1)

    x = tf.image.random_crop(x, size=[crop_size, crop_size, 3])
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    x = tf.clip_by_value(x, 0.0, 1.0)
    x = tf.cast(x, tf.float32)

    tf.debugging.assert_rank(x, 3, message="Output image rank must be 3 after augment")
    tf.debugging.assert_equal(
        tf.shape(x),
        tf.constant([IMG_SIZE, IMG_SIZE, 3], dtype=tf.int32),
        message="Output image shape must be [IMG_SIZE, IMG_SIZE, 3]"
    )

    return x


def _augment_one(args):
    path, image = args
    return strong_augment(path, image)


def target_view(path_batch, image_batch):
    image_batch = tf.cast(image_batch, tf.float32)
    return tf.map_fn(
        _augment_one,
        (path_batch, image_batch),
        fn_output_signature=tf.TensorSpec(
            shape=(IMG_SIZE, IMG_SIZE, 3),
            dtype=tf.float32
        )
    )


def anchor_views(path_batch, image_batch, num_views=2):
    image_batch = tf.cast(image_batch, tf.float32)
    views = []
    for _ in range(num_views):
        views.append(
            tf.map_fn(
                _augment_one,
                (path_batch, image_batch),
                fn_output_signature=tf.TensorSpec(
                    shape=(IMG_SIZE, IMG_SIZE, 3),
                    dtype=tf.float32
                )
            )
        )
    return views
""")


# ============================================================
# 4. model.py
# ============================================================
with open(os.path.join(MODULE_DIR, 'model.py'), 'w', encoding='utf-8') as f:
    f.write(r"""import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = 224
PATCH_SIZE = 16
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2
PROJECTION_DIM = 384
TRANSFORMER_LAYERS = 8
NUM_HEADS = 6
MLP_DIM = PROJECTION_DIM * 4

class Patches(layers.Layer):
    def __init__(self, patch_size=16, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        patch_dim = patches.shape[-1]
        return tf.reshape(patches, [batch_size, -1, patch_dim])

class PatchEmbed(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.proj = layers.Dense(projection_dim)
        self.pos = layers.Embedding(input_dim=num_patches + 1, output_dim=projection_dim)

    def call(self, patch_tokens):
        n = tf.shape(patch_tokens)[1]
        positions = tf.range(1, n + 1)
        return self.proj(patch_tokens) + self.pos(positions)

class TransformerBlock(layers.Layer):
    def __init__(self, dim, num_heads, mlp_dim, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=dim // num_heads,
            dropout=dropout,
        )
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(dim),
            layers.Dropout(dropout),
        ])

    def call(self, x, training=False):
        x = x + self.attn(self.norm1(x), self.norm1(x), training=training)
        x = x + self.mlp(self.norm2(x), training=training)
        return x

class ViTBackbone(tf.keras.Model):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        projection_dim=384,
        num_heads=6,
        depth=8,
        mlp_dim=1536,
        mask_ratio=0.5,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.projection_dim = projection_dim
        self.mask_ratio = mask_ratio

        self.patches = Patches(patch_size=patch_size)
        self.patch_embed = PatchEmbed(self.num_patches, projection_dim)
        self.cls_token = self.add_weight(
            name='cls_token',
            shape=(1, 1, projection_dim),
            initializer='zeros',
            trainable=True,
        )
        self.cls_pos = self.add_weight(
            name='cls_pos',
            shape=(1, 1, projection_dim),
            initializer='zeros',
            trainable=True,
        )
        self.blocks = [
            TransformerBlock(projection_dim, num_heads, mlp_dim) for _ in range(depth)
        ]
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def random_mask_patch_tokens(self, patch_tokens, training=False):
        if (not training) or (self.mask_ratio <= 0.0):
            return patch_tokens

        batch_size = tf.shape(patch_tokens)[0]
        num_tokens = tf.shape(patch_tokens)[1]
        keep_tokens = tf.cast(
            tf.math.round(tf.cast(num_tokens, tf.float32) * (1.0 - self.mask_ratio)),
            tf.int32
        )
        keep_tokens = tf.maximum(1, keep_tokens)

        noise = tf.random.uniform([batch_size, num_tokens])
        ids_shuffle = tf.argsort(noise, axis=1)
        ids_keep = ids_shuffle[:, :keep_tokens]

        batch_indices = tf.tile(tf.range(batch_size)[:, None], [1, keep_tokens])
        gather_idx = tf.stack([batch_indices, ids_keep], axis=-1)
        kept = tf.gather_nd(patch_tokens, gather_idx)
        return kept

    def call(self, images, training=False, apply_mask=False):
        x = self.patches(images)
        x = self.patch_embed(x)
        if apply_mask:
            x = self.random_mask_patch_tokens(x, training=training)

        batch_size = tf.shape(x)[0]
        cls = tf.tile(self.cls_token, [batch_size, 1, 1]) + self.cls_pos
        x = tf.concat([cls, x], axis=1)

        for blk in self.blocks:
            x = blk(x, training=training)
        x = self.norm(x)

        cls_repr = x[:, 0, :]
        return cls_repr

def build_backbone(input_shape=(IMG_SIZE, IMG_SIZE, 3), mask_ratio=0.5):
    model = ViTBackbone(
        img_size=input_shape[0],
        patch_size=PATCH_SIZE,
        projection_dim=PROJECTION_DIM,
        num_heads=NUM_HEADS,
        depth=TRANSFORMER_LAYERS,
        mlp_dim=MLP_DIM,
        mask_ratio=mask_ratio,
    )
    dummy = tf.zeros((1,) + input_shape)
    _ = model(dummy, training=False, apply_mask=False)
    return model

class ProjectionHead(tf.keras.Model):
    def __init__(self, in_dim=PROJECTION_DIM, hidden_dim=1024, out_dim=256, **kwargs):
        super().__init__(**kwargs)
        self.net = models.Sequential([
            layers.Dense(hidden_dim, activation='gelu'),
            layers.Dense(out_dim),
        ])

    def call(self, x, training=False):
        z = self.net(x, training=training)
        z = tf.math.l2_normalize(z, axis=-1)
        return z

def build_projection_head(out_dim=256):
    head = ProjectionHead(out_dim=out_dim)
    dummy = tf.zeros((1, PROJECTION_DIM))
    _ = head(dummy, training=False)
    return head

class Prototypes(tf.keras.Model):
    def __init__(self, num_prototypes=1024, proj_dim=256, **kwargs):
        super().__init__(**kwargs)
        self.num_prototypes = num_prototypes
        self.proj_dim = proj_dim
        self.q = self.add_weight(
            name='prototype_matrix',
            shape=(num_prototypes, proj_dim),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02),
            trainable=True,
        )

    def call(self, z):
        q = tf.math.l2_normalize(self.q, axis=-1)
        return tf.matmul(z, q, transpose_b=True)

def build_prototypes(num_prototypes=1024, proj_dim=256):
    p = Prototypes(num_prototypes=num_prototypes, proj_dim=proj_dim)
    dummy = tf.zeros((1, proj_dim))
    _ = p(dummy)
    return p
""")

# ============================================================
# 5. loss.py
# ============================================================
with open(os.path.join(MODULE_DIR, 'loss.py'), 'w', encoding='utf-8') as f:
    f.write(r"""import tensorflow as tf

def cross_entropy_from_probs(target_probs, student_logits, student_temp):
    student_log_probs = tf.nn.log_softmax(student_logits / student_temp, axis=-1)
    return -tf.reduce_mean(tf.reduce_sum(target_probs * student_log_probs, axis=-1))

def me_max_regularizer(anchor_probs, eps=1e-6):
    mean_probs = tf.reduce_mean(anchor_probs, axis=0)
    entropy = -tf.reduce_sum(mean_probs * tf.math.log(mean_probs + eps))
    return entropy

def msn_loss(
    student_logits_list,
    teacher_logits,
    student_temp=0.2,
    teacher_temp=0.1,
    lambda_me_max=1.0,
):
    teacher_probs = tf.stop_gradient(tf.nn.softmax(teacher_logits / teacher_temp, axis=-1))

    ce_terms = []
    anchor_probs_all = []

    for logits in student_logits_list:
        ce_terms.append(cross_entropy_from_probs(teacher_probs, logits, student_temp))
        anchor_probs_all.append(tf.nn.softmax(logits / student_temp, axis=-1))

    ce_loss = tf.add_n(ce_terms) / tf.cast(len(ce_terms), tf.float32)

    all_anchor_probs = tf.concat(anchor_probs_all, axis=0)
    memax = me_max_regularizer(all_anchor_probs)

    teacher_entropy = -tf.reduce_mean(
        tf.reduce_sum(teacher_probs * tf.math.log(teacher_probs + 1e-6), axis=-1)
    )
    teacher_max_prob_mean = tf.reduce_mean(tf.reduce_max(teacher_probs, axis=-1))

    prototype_prob_mean = tf.reduce_mean(all_anchor_probs, axis=0)
    prototype_top1_ids = tf.argmax(all_anchor_probs, axis=-1, output_type=tf.int32)
    num_prototypes = tf.shape(all_anchor_probs)[-1]
    prototype_top1_hist = tf.math.bincount(
        prototype_top1_ids,
        minlength=num_prototypes,
        maxlength=num_prototypes,
        dtype=tf.float32,
    )
    prototype_top1_hist = prototype_top1_hist / tf.reduce_sum(prototype_top1_hist)

    total = ce_loss - lambda_me_max * memax
    metrics = {
        'total_loss': total,
        'ce_loss': ce_loss,
        'memax_loss': memax,
        'teacher_entropy': teacher_entropy,
        'teacher_max_prob_mean': teacher_max_prob_mean,
        'prototype_prob_mean': prototype_prob_mean,
        'prototype_top1_hist': prototype_top1_hist,
    }
    return total, metrics
""")

# # ============================================================
# # 6. train.py original
# # ============================================================
# with open(os.path.join(MODULE_DIR, 'train.py'), 'w', encoding='utf-8') as f:
#     f.write(r"""import tensorflow as tf
# from loss import msn_loss

# def update_teacher(student_model, teacher_model, momentum=0.996):
#     assert len(student_model.trainable_variables) == len(teacher_model.trainable_variables)
#     tf.debugging.assert_equal(tf.shape(s), tf.shape(t))
#     for s, t in zip(student_model.trainable_variables, teacher_model.trainable_variables):
#         t.assign(momentum * t + (1.0 - momentum) * s)

# @tf.function
# def train_step(
#     anchor_views,
#     target_view,
#     student_backbone,
#     teacher_backbone,
#     student_head,
#     teacher_head,
#     prototypes,
#     optimizer,
#     student_temp=0.2,
#     teacher_temp=0.15,
#     lambda_me_max=1.0,
#     ema_momentum=0.996,
# ):
#     with tf.GradientTape() as tape:
#         student_logits_list = []
#         for view in anchor_views:
#             s_feat = student_backbone(view, training=True, apply_mask=True)
#             s_z = student_head(s_feat, training=True)
#             s_logits = prototypes(s_z)
#             student_logits_list.append(s_logits)

#         t_feat = teacher_backbone(target_view, training=False, apply_mask=False)
#         t_z = teacher_head(t_feat, training=False)
#         #teacher_logits = prototypes(tf.stop_gradient(t_z))
#         teacher_logits = prototypes(t_z)

#         total_loss, metrics = msn_loss(
#             student_logits_list=student_logits_list,
#             teacher_logits=teacher_logits,
#             student_temp=student_temp,
#             teacher_temp=teacher_temp,
#             lambda_me_max=lambda_me_max,
#         )

#     train_vars = (
#         student_backbone.trainable_variables
#         + student_head.trainable_variables
#         + prototypes.trainable_variables
#     )
#     grads = tape.gradient(total_loss, train_vars)
#     optimizer.apply_gradients(zip(grads, train_vars))

#     update_teacher(student_backbone, teacher_backbone, momentum=ema_momentum)
#     update_teacher(student_head, teacher_head, momentum=ema_momentum)

#     return metrics
# """)

# ============================================================
# 6. train.py new
# ============================================================
with open(os.path.join(MODULE_DIR, 'train.py'), 'w', encoding='utf-8') as f:
    f.write(r"""import tensorflow as tf
from loss import msn_loss

def update_teacher(student_model, teacher_model, momentum=0.996):
    student_vars = student_model.trainable_variables
    teacher_vars = teacher_model.trainable_variables

    assert len(student_vars) == len(teacher_vars), (
        f"Teacher/Student variable count mismatch: "
        f"{len(student_vars)} vs {len(teacher_vars)}"
    )

    for s, t in zip(student_vars, teacher_vars):
        tf.debugging.assert_equal(
            tf.shape(s),
            tf.shape(t),
            message="Teacher/Student variable shape mismatch in EMA update"
        )
        t.assign(momentum * t + (1.0 - momentum) * s)


@tf.function
def train_step(
    anchor_views,
    target_view,
    student_backbone,
    teacher_backbone,
    student_head,
    teacher_head,
    prototypes,
    optimizer,
    student_temp=0.2,
    teacher_temp=0.15,
    lambda_me_max=1.0,
    ema_momentum=0.996,
):
    # 先做基本 shape 檢查
    for i, view in enumerate(anchor_views):
        tf.debugging.assert_rank(view, 4, message=f"anchor_views[{i}] must be rank 4: [B,H,W,C]")
        tf.debugging.assert_equal(
            tf.shape(view)[-1], 3,
            message=f"anchor_views[{i}] last channel must be 3"
        )

    tf.debugging.assert_rank(target_view, 4, message="target_view must be rank 4: [B,H,W,C]")
    tf.debugging.assert_equal(
        tf.shape(target_view)[-1], 3,
        message="target_view last channel must be 3"
    )

    with tf.GradientTape() as tape:
        student_logits_list = []

        for view in anchor_views:
            s_feat = student_backbone(view, training=True, apply_mask=True)
            s_z = student_head(s_feat, training=True)
            s_logits = prototypes(s_z)
            student_logits_list.append(s_logits)

        t_feat = teacher_backbone(target_view, training=False, apply_mask=False)
        t_z = teacher_head(t_feat, training=False)

        # teacher branch 作為 target，不進 optimizer；真正 stop_gradient 在 loss.py 的 teacher_probs
        teacher_logits = prototypes(t_z)

        total_loss, metrics = msn_loss(
            student_logits_list=student_logits_list,
            teacher_logits=teacher_logits,
            student_temp=student_temp,
            teacher_temp=teacher_temp,
            lambda_me_max=lambda_me_max,
        )

    train_vars = (
        student_backbone.trainable_variables
        + student_head.trainable_variables
        + prototypes.trainable_variables
    )

    grads = tape.gradient(total_loss, train_vars)

    # 避免某些 grad 是 None 時 apply_gradients 出問題
    grads_and_vars = [
        (g, v) for g, v in zip(grads, train_vars) if g is not None
    ]
    optimizer.apply_gradients(grads_and_vars)

    update_teacher(student_backbone, teacher_backbone, momentum=ema_momentum)
    update_teacher(student_head, teacher_head, momentum=ema_momentum)

    return metrics
""")


# # ============================================================
# # 7. Import modules
# # ============================================================
# import dataset
# import augment
# import model
# import loss
# import train
# from google.colab import runtime
# importlib.reload(dataset)
# importlib.reload(augment)
# importlib.reload(model)
# importlib.reload(loss)
# importlib.reload(train)
# print(dataset.__file__)

# print(f"Modules reloaded from: {MODULE_DIR}")

# # ============================================================
# # 8. Hyperparameters
# # ============================================================
# BATCH_SIZE = 4
# EPOCHS = 2
# NUM_ANCHOR_VIEWS = 2
# MASK_RATIO = 0.5
# NUM_PROTOTYPES = 1024
# PROJ_DIM = 256
# STUDENT_TEMP = 0.2
# TEACHER_TEMP = 0.05
# LAMBDA_ME_MAX = 1.0
# EMA_MOMENTUM = 0.996
# LEARNING_RATE = 1e-4
# DEBUG_PRINT_EVERY = 100  # 可改成 200

# # ============================================================
# # 9. Dataset
# # ============================================================






# # 記錄Main 開始時間
# total_BD_start_time = time.time()
# print(f"\n>>> Total Building_dataset process starts at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(total_BD_start_time))}")
# print(f"Building dataset from: {IMAGE_DIR}")
# train_dataset = dataset.build_dataset(IMAGE_DIR, BATCH_SIZE, split_type='train')
# for batch in train_dataset.take(1):
#     print("batch shape:", batch.shape)
#     print("batch dtype:", batch.dtype)
# total_BD_end_time = time.time()
# total_BD_duration = total_BD_end_time - total_BD_start_time
# print(f"\n<<< Total Building_dataset process ends at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(total_BD_end_time))}")
# print(f"\nTotal Building_dataset duration: {total_BD_duration:.2f} seconds ({total_BD_duration / 60:.2f} minutes)")
# # ============================================================
# # 10. Models
# # ============================================================
# with tf.device('/GPU:0' if DEVICE == 'GPU' else '/CPU:0'):
#     print(f"Building models on {DEVICE}...")
#     student_backbone = model.build_backbone(mask_ratio=MASK_RATIO)
#     teacher_backbone = model.build_backbone(mask_ratio=MASK_RATIO)

#     student_head = model.build_projection_head(out_dim=PROJ_DIM)
#     teacher_head = model.build_projection_head(out_dim=PROJ_DIM)

#     prototypes = model.build_prototypes(num_prototypes=NUM_PROTOTYPES, proj_dim=PROJ_DIM)

#     # initialize teacher = student
#     for s, t in zip(student_backbone.weights, teacher_backbone.weights):
#         t.assign(s)
#     for s, t in zip(student_head.weights, teacher_head.weights):
#         t.assign(s)

#     optimizer = tf.keras.optimizers.AdamW(
#         learning_rate=LEARNING_RATE,
#         weight_decay=1e-4,
#     )

#     # ========================================================
#     # 11. Training loop
#     # ========================================================
#     print("Training start...")
#     global_step = 0
#     for epoch in range(EPOCHS):
#         t0 = time.time()
#         print(f"\n>>> Epoch {epoch+1} started at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(t0))}")

#         metric_sums = {
#             'total_loss': 0.0,
#             'ce_loss': 0.0,
#             'memax_loss': 0.0,
#             'teacher_entropy': 0.0,
#             'teacher_max_prob_mean': 0.0,
#         }
#         steps = 0

#         for batch in train_dataset:
#             anchors = augment.anchor_views(batch, num_views=NUM_ANCHOR_VIEWS)
#             target = augment.target_view(batch)

#             metrics = train.train_step(
#                 anchor_views=anchors,
#                 target_view=target,
#                 student_backbone=student_backbone,
#                 teacher_backbone=teacher_backbone,
#                 student_head=student_head,
#                 teacher_head=teacher_head,
#                 prototypes=prototypes,
#                 optimizer=optimizer,
#                 student_temp=STUDENT_TEMP,
#                 teacher_temp=TEACHER_TEMP,
#                 lambda_me_max=LAMBDA_ME_MAX,
#                 ema_momentum=EMA_MOMENTUM,
#             )

#             for k in metric_sums:
#                 metric_sums[k] += float(metrics[k])

#             steps += 1
#             global_step += 1

#             if global_step % DEBUG_PRINT_EVERY == 0:
#                 proto_prob_mean = metrics['prototype_prob_mean'].numpy()
#                 proto_top1_hist = metrics['prototype_top1_hist'].numpy()

#                 topk = min(10, len(proto_prob_mean))
#                 top_prob_idx = np.argsort(-proto_prob_mean)[:topk]
#                 top_top1_idx = np.argsort(-proto_top1_hist)[:topk]

#                 active_prob = int(np.sum(proto_prob_mean > (1.0 / NUM_PROTOTYPES)))
#                 active_top1 = int(np.sum(proto_top1_hist > 0.0))

#                 print(
#                     f"[step {global_step:06d}] "
#                     f"total={float(metrics['total_loss']):.4f} | "
#                     f"ce={float(metrics['ce_loss']):.4f} | "
#                     f"memax={float(metrics['memax_loss']):.4f} | "
#                     f"teacher_H={float(metrics['teacher_entropy']):.4f} | "
#                     f"teacher_max_p={float(metrics['teacher_max_prob_mean']):.4f}"
#                 )
#                 print(
#                     "  prototype_prob_mean top10: "
#                     + ", ".join([f"{idx}:{proto_prob_mean[idx]:.4f}" for idx in top_prob_idx])
#                 )
#                 print(
#                     "  prototype_top1_hist top10: "
#                     + ", ".join([f"{idx}:{proto_top1_hist[idx]:.4f}" for idx in top_top1_idx])
#                 )
#                 print(
#                     f"  active_prototypes(prob>1/K)={active_prob}/{NUM_PROTOTYPES} | "
#                     f"active_prototypes(top1>0)={active_top1}/{NUM_PROTOTYPES}"
#                 )

#         if steps == 0:
#             print(f"Epoch {epoch+1}: no data.")
#             continue

#         msg = (
#             f"Epoch {epoch+1} | "
#             f"total={metric_sums['total_loss']/steps:.4f} | "
#             f"ce={metric_sums['ce_loss']/steps:.4f} | "
#             f"memax={metric_sums['memax_loss']/steps:.4f} | "
#             f"teacher_H={metric_sums['teacher_entropy']/steps:.4f} | "
#             f"teacher_max_p={metric_sums['teacher_max_prob_mean']/steps:.4f} | "
#             f"time={time.time()-t0:.1f}s"
#         )
#         print(msg)

#     # ========================================================
#     # 12. Save
#     # ========================================================
#     student_backbone.save(os.path.join(MODEL_SAVE_DIR, 'msn_closer_student_backbone.keras'))
#     student_head.save(os.path.join(MODEL_SAVE_DIR, 'msn_closer_student_head.keras'))

#     proto_weights = prototypes.get_weights()
#     np.save(os.path.join(MODEL_SAVE_DIR, 'msn_closer_prototypes.npy'), proto_weights[0])

# print("Saved backbone/head/prototypes.")
# print(f"\n>>> Trainning End at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time()))}")
# runtime.unassign()

# ============================================================
# 7. Import modules
# ============================================================
import dataset
import augment
import model
import loss
import train
from google.colab import runtime
importlib.reload(dataset)
importlib.reload(augment)
importlib.reload(model)
importlib.reload(loss)
importlib.reload(train)
print(dataset.__file__)

print(f"Modules reloaded from: {MODULE_DIR}")

# ============================================================
# 8. Hyperparameters
# ============================================================
BATCH_SIZE = 4
EPOCHS = 2
NUM_ANCHOR_VIEWS = 2
MASK_RATIO = 0.5
NUM_PROTOTYPES = 1024
PROJ_DIM = 256
STUDENT_TEMP = 0.2
TEACHER_TEMP = 0.15   # 原本 0.05，先改高避免 early collapse
LAMBDA_ME_MAX = 1.0
EMA_MOMENTUM = 0.996
LEARNING_RATE = 1e-4
DEBUG_PRINT_EVERY = 100

# ============================================================
# 9. Dataset
# ============================================================
total_BD_start_time = time.time()
print(f"\n>>> Total Building_dataset process starts at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(total_BD_start_time))}")
print(f"Building dataset from: {IMAGE_DIR}")

train_dataset = dataset.build_dataset(
    IMAGE_DIR,
    BATCH_SIZE,
    split_type='train',
    debug_before_augment=False,   # 若要 dataset 階段先印 path/shape，可改 True
)

for path_batch, image_batch in train_dataset.take(1):
    print("path_batch shape:", path_batch.shape)
    print("image_batch shape:", image_batch.shape)
    print("image_batch dtype:", image_batch.dtype)
    tf.print("sample paths =", path_batch, summarize=8)

total_BD_end_time = time.time()
total_BD_duration = total_BD_end_time - total_BD_start_time
print(f"\n<<< Total Building_dataset process ends at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(total_BD_end_time))}")
print(f"\nTotal Building_dataset duration: {total_BD_duration:.2f} seconds ({total_BD_duration / 60:.2f} minutes)")

# ============================================================
# 9.5 Dataset + Augment smoke test（先不訓練，只測資料流）
# ============================================================
RUN_AUGMENT_SMOKE_TEST_ONLY = False # True: 只測 dataset + augment，通過後停止；False: 通過後繼續訓練
print(f"\n<<< SMOKE_TEST process start at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}")
print("\n[SMOKE TEST] Start dataset + augment check...")
# for path_batch, image_batch in train_dataset.take(50):
for step, (path_batch, image_batch) in enumerate(train_dataset.take(50), start=1):
    tf.print(f"\n[SMOKE TEST BATCH {step}]")
    # anchors = augment.anchor_views(path_batch, image_batch, num_views=NUM_ANCHOR_VIEWS)
    # target = augment.target_view(path_batch, image_batch)
    # tf.print("anchor[0] shape =", tf.shape(anchors[0]))
    # tf.print("target shape =", tf.shape(target))

    tf.print("[SMOKE TEST] path_batch =", path_batch, summarize=8)
    tf.print("[SMOKE TEST] image_batch shape =", tf.shape(image_batch))
    tf.print("[SMOKE TEST] image_batch dtype =", image_batch.dtype)

    anchors = augment.anchor_views(path_batch, image_batch, num_views=NUM_ANCHOR_VIEWS)
    target = augment.target_view(path_batch, image_batch)

    tf.print("[SMOKE TEST] anchor[0] shape =", tf.shape(anchors[0]))
    tf.print("[SMOKE TEST] target shape =", tf.shape(target))

    tf.debugging.assert_rank(anchors[0], 4, message="anchor[0] must be rank 4 [B,H,W,C]")
    tf.debugging.assert_rank(target, 4, message="target must be rank 4 [B,H,W,C]")
    tf.debugging.assert_equal(tf.shape(anchors[0])[-1], 3, message="anchor[0] channel must be 3")
    tf.debugging.assert_equal(tf.shape(target)[-1], 3, message="target channel must be 3")

print("[SMOKE TEST] dataset + augment passed.")
print(f"\n<<< SMOKE_TEST process end at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}")
if RUN_AUGMENT_SMOKE_TEST_ONLY:
    print("[SMOKE TEST] RUN_AUGMENT_SMOKE_TEST_ONLY=True, stop before training.")
    raise SystemExit(0)

# ============================================================
# 10. Models
# ============================================================
with tf.device('/GPU:0' if DEVICE == 'GPU' else '/CPU:0'):
    print(f"Building models on {DEVICE}...")

    student_backbone = model.build_backbone(mask_ratio=MASK_RATIO)
    teacher_backbone = model.build_backbone(mask_ratio=MASK_RATIO)

    student_head = model.build_projection_head(out_dim=PROJ_DIM)
    teacher_head = model.build_projection_head(out_dim=PROJ_DIM)

    prototypes = model.build_prototypes(num_prototypes=NUM_PROTOTYPES, proj_dim=PROJ_DIM)

    # initialize teacher = student
    for s, t in zip(student_backbone.weights, teacher_backbone.weights):
        t.assign(s)
    for s, t in zip(student_head.weights, teacher_head.weights):
        t.assign(s)

    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=1e-4,
    )

    # ========================================================
    # 11. Training loop
    # ========================================================
    print("Training start...")
    global_step = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        print(f"\n>>> Epoch {epoch+1} started at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(t0))}")

        metric_sums = {
            'total_loss': 0.0,
            'ce_loss': 0.0,
            'memax_loss': 0.0,
            'teacher_entropy': 0.0,
            'teacher_max_prob_mean': 0.0,
        }
        steps = 0

        for path_batch, image_batch in train_dataset:
            # augment.py 現在要吃 path_batch + image_batch
            anchors = augment.anchor_views(path_batch, image_batch, num_views=NUM_ANCHOR_VIEWS)
            target = augment.target_view(path_batch, image_batch)

            metrics = train.train_step(
                anchor_views=anchors,
                target_view=target,
                student_backbone=student_backbone,
                teacher_backbone=teacher_backbone,
                student_head=student_head,
                teacher_head=teacher_head,
                prototypes=prototypes,
                optimizer=optimizer,
                student_temp=STUDENT_TEMP,
                teacher_temp=TEACHER_TEMP,
                lambda_me_max=LAMBDA_ME_MAX,
                ema_momentum=EMA_MOMENTUM,
            )

            for k in metric_sums:
                metric_sums[k] += float(metrics[k])

            steps += 1
            global_step += 1

            if global_step % DEBUG_PRINT_EVERY == 0:
                proto_prob_mean = metrics['prototype_prob_mean'].numpy()
                proto_top1_hist = metrics['prototype_top1_hist'].numpy()

                topk = min(10, len(proto_prob_mean))
                top_prob_idx = np.argsort(-proto_prob_mean)[:topk]
                top_top1_idx = np.argsort(-proto_top1_hist)[:topk]

                active_prob = int(np.sum(proto_prob_mean > (1.0 / NUM_PROTOTYPES)))
                active_top1 = int(np.sum(proto_top1_hist > 0.0))

                print(
                    f"[step {global_step:06d}] "
                    f"total={float(metrics['total_loss']):.4f} | "
                    f"ce={float(metrics['ce_loss']):.4f} | "
                    f"memax={float(metrics['memax_loss']):.4f} | "
                    f"teacher_H={float(metrics['teacher_entropy']):.4f} | "
                    f"teacher_max_p={float(metrics['teacher_max_prob_mean']):.4f}"
                )
                print(
                    "  prototype_prob_mean top10: "
                    + ", ".join([f"{idx}:{proto_prob_mean[idx]:.4f}" for idx in top_prob_idx])
                )
                print(
                    "  prototype_top1_hist top10: "
                    + ", ".join([f"{idx}:{proto_top1_hist[idx]:.4f}" for idx in top_top1_idx])
                )
                print(
                    f"  active_prototypes(prob>1/K)={active_prob}/{NUM_PROTOTYPES} | "
                    f"active_prototypes(top1>0)={active_top1}/{NUM_PROTOTYPES}"
                )

        if steps == 0:
            print(f"Epoch {epoch+1}: no data.")
            continue

        msg = (
            f"Epoch {epoch+1} | "
            f"total={metric_sums['total_loss']/steps:.4f} | "
            f"ce={metric_sums['ce_loss']/steps:.4f} | "
            f"memax={metric_sums['memax_loss']/steps:.4f} | "
            f"teacher_H={metric_sums['teacher_entropy']/steps:.4f} | "
            f"teacher_max_p={metric_sums['teacher_max_prob_mean']/steps:.4f} | "
            f"time={time.time()-t0:.1f}s"
        )
        print(msg)

    # ========================================================
    # 12. Save
    # ========================================================
    student_backbone.save(os.path.join(MODEL_SAVE_DIR, 'msn_closer_student_backbone.keras'))
    student_head.save(os.path.join(MODEL_SAVE_DIR, 'msn_closer_student_head.keras'))

    proto_weights = prototypes.get_weights()
    np.save(os.path.join(MODEL_SAVE_DIR, 'msn_closer_prototypes.npy'), proto_weights[0])

print("Saved backbone/head/prototypes.")
print(f"\n>>> Training End at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time()))}")
runtime.unassign()



串流輸出內容已截斷至最後 5000 行。
[AUGMENT INPUT] path = "/content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames/video26/video26_000945.png" | shape = [224 224 3]
[AUGMENT CROP] path = "/content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames/video26/video26_000945.png" | crop_size = 190 | pre_crop_shape = [224 224 3]
[AUGMENT INPUT] path = "/content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames/video26/video26_001655.png" | shape = [224 224 3]
[AUGMENT CROP] path = "/content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames/video26/video26_001655.png" | crop_size = 194 | pre_crop_shape = [224 224 3]
[AUGMENT INPUT] path = "/content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames/video27/video27_000198.png" | shape = [224 224 3]
[AUGMENT CROP] path = "/content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames/video27/video27_000198.png" | crop_size = 189 | pre_crop_shape = [224 224 3]
[AUGMENT INPUT] pat